# Project 92 — Local Test Case Generator
## Generate Tests from Code Snippets and Requirements

**Stack:** Ollama · LangChain · Pydantic · Jupyter

## Project Overview

This notebook builds a **local test-case generator** that takes Python code snippets
and/or natural-language requirements as input, then uses a local LLM (via Ollama) to
produce comprehensive test suites.

Everything runs **locally** — the LLM runs via Ollama, no data leaves your machine.

### What You Will Learn

1. How to generate unit tests from **source code** using prompt engineering
2. How to generate test cases from **natural-language requirements**
3. How to **validate** generated tests by compiling and running them in-memory
4. How to produce **categorized** tests (happy path, edge case, error, boundary)
5. How to evaluate test quality and coverage with structured LLM analysis
6. Practical patterns for integrating LLM-based testing into development workflows

## Prerequisites

| Requirement | Details |
|---|---|
| **Ollama** | Running locally with `qwen3:8b` pulled |
| **Python packages** | `langchain`, `langchain-ollama`, `pydantic` |

```bash
# Pull the model (run in terminal)
ollama pull qwen3:8b
```

In [ ]:
# !pip install -q langchain langchain-ollama pydantic

## Step 1 — Imports and Configuration

In [ ]:
import re
import textwrap
import traceback

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Configuration ──────────────────────────────────────────────
OLLAMA_MODEL = "qwen3:8b"
TEMPERATURE = 0.2

print("Configuration ready.")

## Step 2 — Initialize LLM

We use **Qwen3 8B** (via Ollama) for all test generation tasks.
A low temperature keeps output deterministic and code-safe.

In [ ]:
llm = ChatOllama(model=OLLAMA_MODEL, temperature=TEMPERATURE)

# Connectivity check
test_msg = llm.invoke("Reply with only: OK")
print(f"LLM ready: {test_msg.content.strip()[:20]}")

## Step 3 — Define Sample Code Snippets

We define two self-contained Python code snippets that we will use throughout the
notebook. The first is a **ShoppingCart** class (moderate complexity), the second is
a **PasswordValidator** utility (input-validation focused).

Having two different snippets lets us see how the generator handles different coding
patterns.

In [ ]:
SNIPPET_CART = textwrap.dedent('''\
class ShoppingCart:
    """A simple shopping cart with discount support."""

    def __init__(self):
        self.items = {}
        self.discount_code = None

    def add_item(self, name: str, price: float, quantity: int = 1):
        if price < 0:
            raise ValueError("Price cannot be negative")
        if quantity < 1:
            raise ValueError("Quantity must be at least 1")
        if name in self.items:
            self.items[name]["quantity"] += quantity
        else:
            self.items[name] = {"price": price, "quantity": quantity}

    def remove_item(self, name: str):
        if name not in self.items:
            raise KeyError(f"Item {name} not in cart")
        del self.items[name]

    def get_total(self) -> float:
        total = sum(i["price"] * i["quantity"] for i in self.items.values())
        if self.discount_code == "SAVE10":
            total *= 0.9
        elif self.discount_code == "HALF50":
            total *= 0.5
        return round(total, 2)

    def apply_discount(self, code: str):
        valid_codes = ["SAVE10", "HALF50"]
        if code not in valid_codes:
            raise ValueError(f"Invalid discount code: {code}")
        self.discount_code = code

    def item_count(self) -> int:
        return sum(i["quantity"] for i in self.items.values())
''')

SNIPPET_PASSWORD = textwrap.dedent('''\
import re

def validate_password(password: str) -> dict:
    """Validate a password and return a report.

    Returns a dict with keys: valid (bool), errors (list[str]), strength (str).
    """
    errors = []
    if len(password) < 8:
        errors.append("Must be at least 8 characters")
    if len(password) > 128:
        errors.append("Must be at most 128 characters")
    if not re.search(r"[A-Z]", password):
        errors.append("Must contain an uppercase letter")
    if not re.search(r"[a-z]", password):
        errors.append("Must contain a lowercase letter")
    if not re.search(r"[0-9]", password):
        errors.append("Must contain a digit")
    if not re.search(r"[!@#$%^&*(),.?\\":{}|<>]", password):
        errors.append("Must contain a special character")

    valid = len(errors) == 0
    if not valid:
        strength = "weak"
    elif len(password) >= 16:
        strength = "strong"
    else:
        strength = "medium"

    return {"valid": valid, "errors": errors, "strength": strength}
''')

print("=== Snippet 1: ShoppingCart ===")
print(SNIPPET_CART[:200], "...")
print(f"\n=== Snippet 2: PasswordValidator ===")
print(SNIPPET_PASSWORD[:200], "...")

## Step 4 — Build the Test Generation Chain

The core prompt asks the LLM to produce a **comprehensive pytest test suite**
covering five test categories:

1. **Happy-path** — normal usage that should succeed
2. **Edge cases** — empty input, zero values, boundary lengths
3. **Error cases** — invalid input that should raise exceptions
4. **Boundary tests** — values at exact limits
5. **Integration tests** — multiple operations combined

The prompt requests descriptive test names and at least 12 test functions.

In [ ]:
CODE_TO_TESTS_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a senior test engineer. Generate a comprehensive pytest
test suite for the given Python code.

Requirements:
- Include happy-path, edge-case, error-case, boundary, and integration tests.
- Use descriptive test function names that explain what is being tested.
- Include at least 12 test functions.
- Add a brief docstring to each test.
- Put the source code class/function definition at the top so tests are self-contained.
- Return ONLY valid Python code — no markdown fences, no explanation text."""),
    ("human", "Generate a pytest test suite for this code:\n\n{code}")
])

code_test_chain = CODE_TO_TESTS_PROMPT | llm | StrOutputParser()

print("Test generation chain ready.")

## Step 5 — Generate Tests for ShoppingCart

We invoke the chain on the ShoppingCart snippet and inspect the generated test suite.

In [ ]:
cart_tests_raw = code_test_chain.invoke({"code": SNIPPET_CART})

# Strip markdown fences if the model wrapped them anyway
def strip_fences(text: str) -> str:
    """Remove ```python ... ``` wrappers if present."""
    text = text.strip()
    if text.startswith("```"):
        first_nl = text.index("\n")
        text = text[first_nl + 1:]
    if text.endswith("```"):
        text = text[:-3].rstrip()
    return text

cart_tests = strip_fences(cart_tests_raw)

print("GENERATED TEST SUITE — ShoppingCart")
print("=" * 60)
print(cart_tests[:3000])
if len(cart_tests) > 3000:
    print(f"\n... ({len(cart_tests)} chars total)")

## Step 6 — Validate Generated Tests

We validate the generated code in three stages:
1. **Syntax check** — compile the code to catch syntax errors
2. **Exec check** — execute it to verify imports and class/function definitions load
3. **Test count** — count `def test_` functions to verify coverage depth

In [ ]:
def validate_test_code(test_code: str, label: str = "tests") -> dict:
    """Validate generated test code: syntax, execution, test count."""
    result = {"label": label, "syntax_ok": False, "exec_ok": False,
              "test_count": 0, "error": None}

    # 1. Syntax check
    try:
        compile(test_code, "<generated>", "exec")
        result["syntax_ok"] = True
    except SyntaxError as e:
        result["error"] = f"SyntaxError: {e}"
        return result

    # 2. Exec check (imports + definitions only, no test runner)
    try:
        exec(test_code, {})
        result["exec_ok"] = True
    except Exception as e:
        result["error"] = f"{type(e).__name__}: {e}"

    # 3. Count test functions
    result["test_count"] = len(re.findall(r"^def test_", test_code, re.MULTILINE))

    return result


cart_validation = validate_test_code(cart_tests, "ShoppingCart")
print(f"Validation — {cart_validation['label']}")
print(f"  Syntax OK:  {cart_validation['syntax_ok']}")
print(f"  Exec OK:    {cart_validation['exec_ok']}")
print(f"  Test count: {cart_validation['test_count']}")
if cart_validation["error"]:
    print(f"  Error:      {cart_validation['error']}")

## Step 7 — Generate Tests for PasswordValidator

Now we run the same chain on the second snippet. This tests how the generator
handles a standalone function (vs. a class) with regex-based validation logic.

In [ ]:
pw_tests_raw = code_test_chain.invoke({"code": SNIPPET_PASSWORD})
pw_tests = strip_fences(pw_tests_raw)

print("GENERATED TEST SUITE — PasswordValidator")
print("=" * 60)
print(pw_tests[:3000])
if len(pw_tests) > 3000:
    print(f"\n... ({len(pw_tests)} chars total)")

pw_validation = validate_test_code(pw_tests, "PasswordValidator")
print(f"\nValidation — {pw_validation['label']}")
print(f"  Syntax OK:  {pw_validation['syntax_ok']}")
print(f"  Exec OK:    {pw_validation['exec_ok']}")
print(f"  Test count: {pw_validation['test_count']}")
if pw_validation["error"]:
    print(f"  Error:      {pw_validation['error']}")

## Step 8 — Requirements-Based Test Generation

Often you don't have code yet — you have **requirements** or a specification.
This step takes natural-language requirements and generates test cases that
the code should eventually satisfy.

This is useful for **test-driven development (TDD)**: write tests first from
requirements, then implement code to make them pass.

In [ ]:
REQUIREMENTS = """
Feature: User Registration System

1. Users must provide an email, username, and password.
2. Email must be a valid email format (contains @ and a domain).
3. Username must be 3-20 characters, alphanumeric and underscores only.
4. Password must be at least 8 characters with one uppercase, one digit.
5. Duplicate emails are not allowed — raise ValueError.
6. Duplicate usernames are not allowed — raise ValueError.
7. Successful registration returns a dict with user_id, email, username, created_at.
8. user_id must be a unique string (UUID format).
9. created_at must be a valid ISO 8601 datetime string.
10. The system should handle leading/trailing whitespace in all inputs.
"""

REQ_TO_TESTS_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a senior QA engineer. Given a set of feature requirements,
generate a comprehensive pytest test suite.

Rules:
- Write tests ONLY — do not implement the feature.
- Assume a function `register_user(email, username, password)` exists.
- Test each requirement explicitly. Name tests after the requirement.
- Include positive tests, negative tests, and edge cases.
- Include at least 15 test functions.
- Add a brief docstring to each test.
- Return ONLY valid Python code — no markdown fences, no explanation."""),
    ("human", "Generate tests from these requirements:\n\n{requirements}")
])

req_test_chain = REQ_TO_TESTS_PROMPT | llm | StrOutputParser()

print("Requirements:")
print(REQUIREMENTS)

In [ ]:
req_tests_raw = req_test_chain.invoke({"requirements": REQUIREMENTS})
req_tests = strip_fences(req_tests_raw)

print("GENERATED TEST SUITE — From Requirements")
print("=" * 60)
print(req_tests[:4000])
if len(req_tests) > 4000:
    print(f"\n... ({len(req_tests)} chars total)")

# Syntax-only validation (code under test doesn't exist yet)
try:
    compile(req_tests, "<req_tests>", "exec")
    print("\n✓ Syntax OK")
except SyntaxError as e:
    print(f"\n✗ SyntaxError: {e}")

req_test_count = len(re.findall(r"^def test_", req_tests, re.MULTILINE))
print(f"Generated {req_test_count} test functions from requirements")

## Step 9 — Test Categorization

We ask the LLM to **analyze** generated test code and classify each test function
into one of five categories:

| Category | Description |
|---|---|
| **happy_path** | Normal usage that should succeed |
| **edge_case** | Boundary or unusual-but-valid inputs |
| **error_case** | Invalid inputs that should raise exceptions |
| **boundary** | Values at exact limits (min, max) |
| **integration** | Multiple operations combined in sequence |

This gives visibility into whether the generated suite has balanced coverage.

In [ ]:
CATEGORIZE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Analyze the test functions below and classify each into exactly
one category: happy_path, edge_case, error_case, boundary, or integration.

Return the result as a simple text table with columns:
  function_name | category | one_line_description

Include ALL test functions. Do not skip any. No extra commentary."""),
    ("human", "{tests}")
])

categorize_chain = CATEGORIZE_PROMPT | llm | StrOutputParser()

cart_categories = categorize_chain.invoke({"tests": cart_tests})
print("TEST CATEGORIES — ShoppingCart")
print("=" * 60)
print(cart_categories)

In [ ]:
pw_categories = categorize_chain.invoke({"tests": pw_tests})
print("TEST CATEGORIES — PasswordValidator")
print("=" * 60)
print(pw_categories)

## Step 10 — Coverage Gap Analysis

We ask the LLM to identify **missing** test scenarios — things the generated
suite did not cover. This is a quality check: even a good test generator can
miss subtle scenarios.

In [ ]:
GAP_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a test coverage reviewer. Given the source code and the
existing test suite, identify test scenarios that are MISSING.

For each gap, provide:
1. The scenario description
2. Why it matters
3. A suggested test name

Be specific and practical. List at least 3 gaps."""),
    ("human", "Source code:\n{code}\n\nExisting tests:\n{tests}")
])

gap_chain = GAP_PROMPT | llm | StrOutputParser()

cart_gaps = gap_chain.invoke({"code": SNIPPET_CART, "tests": cart_tests})
print("COVERAGE GAPS — ShoppingCart")
print("=" * 60)
print(cart_gaps)

In [ ]:
pw_gaps = gap_chain.invoke({"code": SNIPPET_PASSWORD, "tests": pw_tests})
print("COVERAGE GAPS — PasswordValidator")
print("=" * 60)
print(pw_gaps)

## Step 11 — Generate Additional Tests for Gaps

We take the coverage gap analysis and ask the LLM to generate the **missing**
tests. This closes the loop: generate → validate → analyze gaps → fill gaps.

In [ ]:
FILL_GAPS_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Generate ONLY the missing pytest test functions described below.
Include the source code definition at the top so the tests are self-contained.
Return ONLY valid Python code — no markdown fences, no explanation."""),
    ("human", "Source code:\n{code}\n\nMissing test scenarios:\n{gaps}")
])

fill_chain = FILL_GAPS_PROMPT | llm | StrOutputParser()

extra_cart_raw = fill_chain.invoke({"code": SNIPPET_CART, "gaps": cart_gaps})
extra_cart = strip_fences(extra_cart_raw)

print("ADDITIONAL TESTS — ShoppingCart")
print("=" * 60)
print(extra_cart[:2000])

extra_validation = validate_test_code(extra_cart, "ShoppingCart extras")
print(f"\nValidation: syntax_ok={extra_validation['syntax_ok']}, "
      f"exec_ok={extra_validation['exec_ok']}, "
      f"test_count={extra_validation['test_count']}")
if extra_validation["error"]:
    print(f"  Error: {extra_validation['error']}")

## Step 12 — Full Pipeline Summary

Let's summarize everything generated across the notebook.

In [ ]:
print("TEST GENERATION SUMMARY")
print("=" * 60)
print()

rows = [
    ("ShoppingCart (code→tests)", cart_validation["test_count"],
     cart_validation["syntax_ok"], cart_validation["exec_ok"]),
    ("PasswordValidator (code→tests)", pw_validation["test_count"],
     pw_validation["syntax_ok"], pw_validation["exec_ok"]),
    ("User Registration (req→tests)", req_test_count, True, "N/A"),
    ("ShoppingCart gap-fill", extra_validation["test_count"],
     extra_validation["syntax_ok"], extra_validation["exec_ok"]),
]

print(f"{'Suite':<35} {'Tests':>5}  {'Syntax':>6}  {'Exec':>5}")
print("-" * 60)
total = 0
for name, count, syn, exc in rows:
    total += count
    print(f"{name:<35} {count:>5}  {str(syn):>6}  {str(exc):>5}")
print("-" * 60)
print(f"{'TOTAL':<35} {total:>5}")

## Evaluation Summary

| Dimension | How We Evaluated |
|---|---|
| **Syntax validity** | Every generated suite was compiled with `compile()` |
| **Execution validity** | Suites were `exec()`'d to verify definitions load |
| **Test count** | Counted `def test_` functions — target ≥ 10 per suite |
| **Category balance** | LLM categorized tests across 5 categories (Step 9) |
| **Coverage completeness** | LLM gap analysis identified missing scenarios (Step 10) |
| **Gap-fill quality** | Additional tests generated and validated (Step 11) |
| **Requirements → tests** | Verified tests can be generated from spec alone (Step 8) |

### Known Limitations

- **Test correctness**: Syntax validity does not guarantee logical correctness.
  A test can compile and run but assert the wrong thing.
- **Flaky assertions**: LLM-generated assertions sometimes use hard-coded values
  that depend on implementation details.
- **Markdown leakage**: Despite prompting, the LLM occasionally wraps output in
  code fences — the `strip_fences()` helper mitigates this.
- **No mutation testing**: We don't verify that tests actually *detect* bugs.
- **Model dependency**: Test quality varies with model size and capability.

## How to Improve This Project

1. **Mutation testing** — introduce deliberate bugs in the source code and verify
   that the generated tests catch them.
2. **Iterative refinement** — if tests fail validation, automatically re-prompt the
   LLM with the error message to fix the output.
3. **Structured output** — use Pydantic models with `with_structured_output()` to get
   JSON-formatted test metadata alongside code.
4. **Multi-language support** — extend the prompts to generate tests for JavaScript,
   TypeScript, or Go code.
5. **File-based workflow** — read `.py` files from disk, generate test files, and
   write them to a `tests/` directory.
6. **Property-based testing** — prompt the LLM to generate Hypothesis strategies
   for property-based tests.

## What You Learned

- **Code → tests** — generating comprehensive test suites from Python source code
- **Requirements → tests** — producing TDD-style tests from natural-language specs
- **Test validation** — compiling and executing generated code to catch errors
- **Test categorization** — classifying tests by type for coverage visibility
- **Coverage gap analysis** — finding missing scenarios and generating fill-in tests
- **Prompt engineering** — crafting system prompts for consistent, code-only output